In [15]:
import tensorflow as tf
from tensorflow.keras.models import load_model
import pandas as pd
import pickle
import pandas as pd
import numpy as np

In [49]:
## load the trained model ,scalerpickle,onehot
model=load_model('model.h5')
## load encoder and scaler
with open('/content/one_hot_encode_geo.pkl','rb') as file:
  one_hot_encode_geo=pickle.load(file)

with open("/content/label_encoder_gender.pkl",'rb') as file:
  lebel_encoder_gender=pickle.load(file)

with open('/content/scaler.pkl','rb') as file:
  scaler=pickle.load(file)

In [40]:
## Example input data
input_data={
    'CreditScore':600,
    'Geography':'France',
    'Gender':'Male',
    'Age':40,
    'Tenure':3,
    'Balance':60000,
    'NumOfProducts':2,
    'HasCrCard':1,
    'IsActiveMember':1,
    'EstimatedSalary':50000
}

In [41]:
input_data_df=pd.DataFrame([input_data])

In [42]:
## We stored things in pickle fole to do there task with input data
## we ise transform to particular pickel files
geo_encoded=one_hot_encode_geo.transform([[input_data['Geography']]]).toarray()
geo_encoded_df=pd.DataFrame(geo_encoded,columns=one_hot_encode_geo.get_feature_names_out(['Geography']))
gender_encoded = lebel_encoder_gender.transform([input_data['Gender']])
gender_encoded_df=pd.DataFrame(gender_encoded,columns=['Gender'])

/usr/local/lib/python3.11/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but OneHotEncoder was fitted with feature names
  warnings.warn(


In [43]:
geo_encoded_df

,Geography_France,Geography_Germany,Geography_Spain
0,1.0,0.0,0.0


In [44]:
gender_encoded_df

,Gender
0,1


In [47]:
input_data=pd.concat([input_data_df.reset_index(drop=True),geo_encoded_df],axis=1)

array([1])

In [68]:
input_data_df['Gender']=lebel_encoder_gender.transform(input_data_df['Gender'])

In [69]:
input_data_df

,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary
0,600,France,1,40,3,60000,2,1,1,50000


In [70]:
# concate one hot encoded
input_data_df=pd.concat([input_data_df.drop(['Geography'],axis=1),geo_encoded_df],axis=1)

In [71]:
input_data_df

,CreditScore,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Geography_France,Geography_Germany,Geography_Spain
0,600,1,40,3,60000,2,1,1,50000,1.0,0.0,0.0


In [72]:
## Scaling the input data
input_scaled=scaler.transform(input_data_df)


In [73]:
input_scaled

array([[-0.53598516,  0.91324755,  0.10479359, -0.69539349, -0.25781119,
         0.80843615,  0.64920267,  0.97481699, -0.87683221,  1.00150113,
        -0.57946723, -0.57638802]])

In [76]:
## Predict churn
prediction=model.predict(input_scaled)
prediction

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 39ms/step


array([[0.0197102]], dtype=float32)

In [77]:
prediction_prob=prediction[0][0]

In [78]:
if prediction_prob>0.5:
  print("The customer is likely to churn")
else:
  print("Customer is not likely to churn")

Customer is not likely to churn
